# Forward modeling in Cascadia with Green's functions

This notebook builds synthetic seismograms for the Cascadia subduction zone by the "forward" route of MIT 12.510, Lecture 17 (Section B, equations 7 to 9): **known source + known Green's functions → predicted seismogram**.

It has two parts:

| Part | Source | What it shows |
|---|---|---|
| **1** | 2001 M6.8 Nisqually intraslab earthquake (a real event) | Build a synthetic from six elementary Green's functions, prove that the superposition is exact, then compare the prediction with seismograms recorded in 2001. |
| **2** | Scenario M9.0 full-margin megathrust rupture (a "1700-type" event) | Discretize the real plate interface (Slab2) into subfaults, compute one Green's function per subfault, and sum them with rupture delays. This is equation (7) of the lecture, written out as code. Then reuse the same Green's functions for a second rupture scenario without new computation. |

**Where the Green's functions come from.** The notebook uses the EarthScope **Syngine** service. Syngine extracts waveforms from precomputed AxiSEM wavefield databases with Instaseis, for 1D (radially symmetric) Earth models. Each Syngine request in this notebook is one Green's function evaluation: the ground response at a receiver to a point source of known mechanism and a known moment.

**Runtime and network.** The notebook needs network access to `service.earthscope.org` (Syngine and FDSN), `earthquake.usgs.gov` (ComCat), and `www.sciencebase.gov` (Slab2). Part 2 sends about 100 Syngine requests on the first run. Every response is cached on disk, so a second run is fast and sends no requests.

**Target environment.** GeoLab (Pangeo-based image). The required packages are ObsPy, NumPy, SciPy, Matplotlib, pandas, xarray (with a netCDF backend), and requests. Cartopy is optional; it only improves the maps.

## How to read the annotations

The notebook flags every modeling choice where it is made:

- **[A#] Assumption.** A choice that changes the numbers. Each one names the parameter that controls it, so you can change it.
- **[L#] Limitation.** A known way in which the model differs from the real Earth.
- **[C#] Check.** A cell that tests an assumption and **raises an error** when the test fails. The notebook does not continue silently after a failed check.
- **[R#] Reference.** The literature or service documentation that supports a value. The full list is at the end.

## 0. Environment and configuration

In [ ]:
# Environment check: fail loudly on missing required packages, report optional ones.
import importlib

REQUIRED = ["numpy", "scipy", "matplotlib", "pandas", "xarray", "requests", "obspy"]
OPTIONAL = ["cartopy", "netCDF4", "h5netcdf"]

missing = []
for name in REQUIRED:
    try:
        mod = importlib.import_module(name)
        print(f"{name:12s} {getattr(mod, '__version__', '?')}")
    except ImportError:
        missing.append(name)
if missing:
    raise ImportError(f"Required packages are missing: {missing}. Install them in your GeoLab environment.")

for name in OPTIONAL:
    try:
        mod = importlib.import_module(name)
        print(f"{name:12s} {getattr(mod, '__version__', '?')} (optional)")
    except ImportError:
        print(f"{name:12s} not installed (optional)")

In [ ]:
import hashlib
import io
import json
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import xarray as xr
from scipy import signal

import obspy
from obspy import UTCDateTime, Stream, Trace, read
from obspy.clients.fdsn import Client as FDSNClient
from obspy.clients.syngine import Client as SyngineClient
from obspy.geodetics import gps2dist_azimuth

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAVE_CARTOPY = True
except ImportError:
    HAVE_CARTOPY = False
    print("Cartopy is not available. Maps will use plain longitude/latitude axes.")

### Configuration

All tunable parameters are in the next cell. The assumption labels ([A#]) in later sections refer back to these names.

In [ ]:
# ---- Services -------------------------------------------------------------
FDSN_PROVIDER = "EARTHSCOPE"      # ObsPy maps this key to https://service.earthscope.org
SYNGINE_TIMEOUT_S = 300
MAX_WORKERS = 4                   # Parallel Syngine requests. Keep this small; it is a shared public service.

# ---- Green's function database ---------------------------------------------
# [A1] 1D Earth model. ak135f_5s resolves periods down to about 5 s, which is
#      shorter than every period used below. See [L1] for what a 1D model omits.
MODEL = "ak135f_5s"
M_UNIT = 1.0e18                   # Moment (N m) of each elementary source request. Results are scaled linearly.

# ---- Local storage ----------------------------------------------------------
CACHE_DIR = Path("cache_cascadia_gf")
DATA_DIR = Path("data_cascadia")
CACHE_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# ---- Part 1: 2001 Nisqually -------------------------------------------------
COMCAT_EVENT_ID = "uw10530748"    # USGS ComCat ID of the 2001 Nisqually earthquake [R10]
MT_SOURCE = "comcat"              # "comcat" (moment tensor from ComCat) or "ichinose2004" (published fault plane, see [A3])
P1_MINRADIUS_DEG = 1.0
P1_MAXRADIUS_DEG = 10.0
P1_N_STATIONS = 6                 # One station per azimuth sector, nearest first
P1_DURATION_S = 900.0
P1_PERIOD_BAND_S = (20.0, 100.0)  # [A2] Long-period band for the data comparison
P1_PAD_S = 300.0                  # Extra data before and after, for response removal

# ---- Part 2: M9 scenario ----------------------------------------------------
SLAB2_ITEM_ID = "5aa312cde4b0b1c392ea3ef5"  # ScienceBase item: Slab2, Cascadia region [R7]
LAT_MIN, LAT_MAX, DLAT = 40.4, 50.0, 0.4    # [A4] Along-strike extent and subfault size (about 44 km)
DEPTH_EDGES_KM = [0.0, 7.0, 13.0, 19.0, 25.0]  # [A5] Down-dip rows; 25 km is the down-dip limit of slip
SLIP_DOWNDIP_WEIGHTS = [1.0, 1.0, 0.7, 0.35]   # [A6] Relative slip per down-dip row (more slip at shallow depth)
RANDOM_SLIP_SIGMA = 0.3           # [A6] Log-normal slip heterogeneity; 0 disables it
RANDOM_SEED = 1700
TARGET_MW = 9.0                   # [A7]
RIGIDITY_PA = 3.0e10              # [A7]
RAKE_DEG = 90.0                   # [A8] Pure thrust on every subfault
RUPTURE_VELOCITY_KMS = 2.5        # [A9]
SLIP_VELOCITY_MS = 0.65           # [A10] Background slip velocity [R8]
MAX_RISE_TIME_S = 30.0            # [A10] Upper limit of the rise time [R8]
HYPOCENTERS = {                   # [A11] (latitude, depth in km); nearest subfault is used
    "south": (41.0, 15.0),
    "north": (49.3, 15.0),
}
P2_REQUEST_S = 1500.0             # Length of each Green's function
P2_PLOT_S = 1000.0
P2_MIN_PERIOD_S = 30.0            # [A12] Low-pass corner for the scenario, set by subfault size

# [A13] Receivers are city-center coordinates (not seismic stations).
RECEIVERS = pd.DataFrame(
    [
        ("CRSC", "Crescent City, CA", 41.756, -124.202),
        ("EUGN", "Eugene, OR",        44.052, -123.087),
        ("PDX",  "Portland, OR",      45.515, -122.679),
        ("ABRD", "Aberdeen, WA",      46.976, -123.815),
        ("SEA",  "Seattle, WA",       47.606, -122.332),
        ("VICT", "Victoria, BC",      48.428, -123.366),
    ],
    columns=["code", "name", "lat", "lon"],
)

### Service helpers

`syngine_bulk` sends one Syngine request (one source, many receivers) and caches the result under a hash of every request parameter. When a request fails three times, the function raises the error with the full parameter set. It never returns an empty or partial result.

In [ ]:
syn = SyngineClient(timeout=SYNGINE_TIMEOUT_S)
fdsn = FDSNClient(FDSN_PROVIDER, timeout=180)
print("Syngine endpoint:", syn.base_url)
print("FDSN endpoint:   ", fdsn.base_url)


def _cache_key(payload):
    blob = json.dumps(payload, sort_keys=True, default=str).encode()
    return hashlib.sha1(blob).hexdigest()[:24]


def syngine_bulk(bulk, retries=3, **params):
    # One source, many receivers. Returns an ObsPy Stream. Cached on disk.
    params = {k: v for k, v in params.items() if v is not None}
    path = CACHE_DIR / f"{_cache_key({'bulk': bulk, 'url': syn.base_url, **params})}.mseed"
    if path.exists():
        return read(str(path))
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            st = syn.get_waveforms_bulk(bulk=bulk, **params)
            break
        except Exception as err:          # network or service error
            last_error = err
            time.sleep(5 * attempt)
    else:
        raise RuntimeError(
            f"Syngine request failed after {retries} attempts.\n"
            f"Last error: {last_error!r}\nParameters: {params}"
        ) from last_error
    expected = len(bulk) * len(params.get("components", "ZNE"))
    if len(st) != expected:
        raise RuntimeError(f"Syngine returned {len(st)} traces; expected {expected}. Parameters: {params}")
    st.write(str(path), format="MSEED")
    return st


def pick(st, station, component):
    # Exactly one trace, or an error.
    sel = st.select(station=station, component=component)
    if len(sel) != 1:
        raise KeyError(f"Expected 1 trace for {station}.{component}, found {len(sel)}")
    return sel[0]


# [C0] The chosen model must exist on the service.
available = syn.get_available_models()
names = sorted(available.keys()) if isinstance(available, dict) else sorted(m.get("name", str(m)) for m in available)
if MODEL not in names:
    raise ValueError(f"Model {MODEL!r} is not on the service. Available models: {names}")
info = syn.get_model_info(MODEL)
print(f"Model {MODEL}: shortest period {info.get('period')} s, database dt {info.get('dt')} s, "
      f"length {info.get('length')} s")

## 1. Setting and theory

### Cascadia

The Juan de Fuca plate subducts beneath North America along the Cascadia subduction zone, from northern California to Vancouver Island. Three kinds of damaging earthquakes occur here [R11, R12]:

1. **Megathrust earthquakes** on the plate interface. The last one was on 26 January 1700. Tsunami modeling of Japanese historical records supports a rupture about 1100 km long with an average slip of about 14 m, M0 ≈ 5 × 10²² N m, Mw ≈ 9.0 [R6]. No seismograph has recorded one, so ground-motion estimates depend on simulations such as those of the M9 Project [R8, R9].
2. **Intraslab earthquakes** within the subducting plate, 30 to 70 km deep. The 2001 M6.8 Nisqually earthquake was a normal-faulting event of this type at about 52 km depth beneath southern Puget Sound [R12].
3. **Shallow crustal earthquakes** in the North American plate, for example on the Seattle fault.

Part 1 uses an event of type 2 because modern broadband records of it exist. Part 2 builds a scenario of type 1.

### From the lecture to the code

Lecture 17 writes the displacement as the source convolved with the Green's function and integrated over the source volume (equation 7), and compresses it to u ≈ f · G (equation 9). For an earthquake, the body force f is replaced by an equivalent **moment tensor** M (the representation theorem). For a point source with moment-rate function s(t) of unit area,

$$
u_i(\mathbf{x}, t) \;=\; \sum_{k=1}^{6} M_k \,\big[\, g_{ik}(\mathbf{x}, t) * s(t) \,\big],
\qquad k \in \{M_{rr}, M_{\theta\theta}, M_{\phi\phi}, M_{r\theta}, M_{r\phi}, M_{\theta\phi}\}
$$

where g_ik is the response at component i to a unit step of moment-tensor element k. These six g_ik are **elementary Green's functions**: they depend only on the Earth model and the source-receiver geometry, not on the earthquake.

For a finite fault (Part 2), equation (7) becomes a sum over subfaults j, each delayed by its rupture time T_j:

$$
u_i(\mathbf{x}, t) \;=\; \sum_{j} M_{0,j}\, \big[\, G_{ij}(\mathbf{x}, t - T_j) * s_j(t) \,\big]
$$

where G_ij is the response to a unit-moment double couple at subfault j. The M9 Project used the same structure: a sum of point-source synthetics, each delayed by rupture time and convolved with a slip-velocity function [R8]. The difference is that they used 3D wave propagation and about 600,000 source points.

## 2. Part 1: the 2001 Nisqually earthquake

### 2.1 Source-parameter utilities

Moment tensors use the Syngine component order (Mrr, Mθθ, Mφφ, Mrθ, Mrφ, Mθφ) in N m, with r up, θ south, φ east (the Global CMT convention).

In [ ]:
MT_KEYS = ["Mrr", "Mtt", "Mpp", "Mrt", "Mrp", "Mtp"]


def sdr_to_mt(strike, dip, rake, m0):
    # Aki and Richards (2002) Box 4.4, with x north, y east, z down [R3],
    # then converted to r up, theta south, phi east.
    phi, delta, lam = np.radians([strike, dip, rake])
    mxx = -m0 * (np.sin(delta) * np.cos(lam) * np.sin(2 * phi) + np.sin(2 * delta) * np.sin(lam) * np.sin(phi) ** 2)
    myy = m0 * (np.sin(delta) * np.cos(lam) * np.sin(2 * phi) - np.sin(2 * delta) * np.sin(lam) * np.cos(phi) ** 2)
    mzz = m0 * np.sin(2 * delta) * np.sin(lam)
    mxy = m0 * (np.sin(delta) * np.cos(lam) * np.cos(2 * phi) + 0.5 * np.sin(2 * delta) * np.sin(lam) * np.sin(2 * phi))
    mxz = -m0 * (np.cos(delta) * np.cos(lam) * np.cos(phi) + np.cos(2 * delta) * np.sin(lam) * np.sin(phi))
    myz = -m0 * (np.cos(delta) * np.cos(lam) * np.sin(phi) - np.cos(2 * delta) * np.sin(lam) * np.cos(phi))
    # Mrr = Mzz, Mtt = Mxx, Mpp = Myy, Mrt = Mxz, Mrp = -Myz, Mtp = -Mxy
    return np.array([mzz, mxx, myy, mxz, -myz, -mxy])


def scalar_moment(m):
    # Frobenius-norm definition (Silver and Jordan, 1982) [R14].
    mrr, mtt, mpp, mrt, mrp, mtp = m
    return np.sqrt((mrr**2 + mtt**2 + mpp**2 + 2 * (mrt**2 + mrp**2 + mtp**2)) / 2)


def mw_from_m0(m0):
    return (2.0 / 3.0) * (np.log10(m0) - 9.1)


def m0_from_mw(mw):
    return 10 ** (1.5 * mw + 9.1)


def gcmt_half_duration(m0):
    # Global CMT half-duration scaling, h = 1.05e-8 * M0**(1/3), M0 in dyne cm [R4].
    return 1.05e-8 * (m0 * 1e7) ** (1.0 / 3.0)

### 2.2 The source

The default source is the preferred moment-tensor product in USGS ComCat for event `uw10530748`. The cell prints which catalog supplied it.

**[A3]** If ComCat does not return a moment tensor, the cell stops and asks you to choose. Set `MT_SOURCE = "ichinose2004"` to use the published finite-fault setup of Ichinose et al. (2004) instead: hypocenter 47.14°N, 122.71°W, 56 km; the east-dipping plane with strike 350° and dip 70°; M0 = 1.7 × 10¹⁹ N m [R5]. They allowed the rake to vary between −45° and −135°; the value −90° used here is the midpoint of that range, not a published result.

In [ ]:
def fetch_comcat_moment_tensor(event_id):
    url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    r = requests.get(url, params={"eventid": event_id, "format": "geojson"}, timeout=60)
    r.raise_for_status()
    ev = r.json()
    products = ev["properties"].get("products", {})
    mts = products.get("moment-tensor", [])
    if not mts:
        raise LookupError(
            f"ComCat has no moment-tensor product for {event_id}. "
            "Set MT_SOURCE = 'ichinose2004' in the configuration cell to use the published fault plane."
        )
    mt = max(mts, key=lambda p: p.get("preferredWeight", 0))
    p = mt["properties"]
    tensor = np.array([float(p[f"tensor-{k.lower()}"]) for k in MT_KEYS])
    lon0, lat0, dep0 = ev["geometry"]["coordinates"]
    source = {
        "label": f"ComCat moment tensor (source: {mt.get('source')}, method: {p.get('beachball-source', 'n/a')})",
        "lat": float(p.get("derived-latitude", lat0)),
        "lon": float(p.get("derived-longitude", lon0)),
        "depth_km": float(p.get("derived-depth", dep0)),
        "time": UTCDateTime(p["derived-eventtime"]) if "derived-eventtime" in p
                else UTCDateTime(ev["properties"]["time"] / 1000.0),
        "mt": tensor,
        "half_duration_s": float(p["sourcetime-duration"]) / 2 if "sourcetime-duration" in p else None,
    }
    used_derived = [k for k in ("derived-latitude", "derived-longitude", "derived-depth", "derived-eventtime") if k in p]
    source["label"] += f"; centroid fields used: {used_derived or 'none (hypocenter used)'}"
    return source


if MT_SOURCE == "comcat":
    SRC = fetch_comcat_moment_tensor(COMCAT_EVENT_ID)
elif MT_SOURCE == "ichinose2004":
    SRC = {
        "label": "Ichinose et al. (2004) fault plane; rake -90 deg is an assumption [A3]",
        "lat": 47.14, "lon": -122.71, "depth_km": 56.0,
        "time": UTCDateTime("2001-02-28T18:54:32.78"),
        "mt": sdr_to_mt(350.0, 70.0, -90.0, 1.7e19),
        "half_duration_s": None,
    }
else:
    raise ValueError(f"Unknown MT_SOURCE {MT_SOURCE!r}")

SRC["m0"] = scalar_moment(SRC["mt"])
if SRC["half_duration_s"] is None:
    SRC["half_duration_s"] = gcmt_half_duration(SRC["m0"])
    hd_note = "from the GCMT scaling relation [R4]"
else:
    hd_note = "from the catalog"

print(SRC["label"])
print(f"Location {SRC['lat']:.3f} N, {SRC['lon']:.3f} E, depth {SRC['depth_km']:.1f} km, time {SRC['time']}")
print("Moment tensor (N m):", dict(zip(MT_KEYS, [f"{x:.3e}" for x in SRC["mt"]])))
print(f"M0 = {SRC['m0']:.3e} N m, Mw = {mw_from_m0(SRC['m0']):.2f}")
print(f"Half duration = {SRC['half_duration_s']:.1f} s ({hd_note})")
print(f"Mrr is {'negative' if SRC['mt'][0] < 0 else 'positive'}: "
      f"{'normal faulting, as expected for Nisqually' if SRC['mt'][0] < 0 else 'check this source; Nisqually was a normal-faulting event'}")

### 2.3 Stations

The cell asks the EarthScope FDSN station service for broadband (`BH?`) stations that were operating at the event time, between `P1_MINRADIUS_DEG` and `P1_MAXRADIUS_DEG` from the source. It then keeps the nearest station in each of `P1_N_STATIONS` azimuth sectors, so that the comparison samples different parts of the radiation pattern.

Many stations that operate today did not exist in 2001, so the selection depends on what the archive held at that time.

In [ ]:
inv_all = fdsn.get_stations(
    latitude=SRC["lat"], longitude=SRC["lon"],
    minradius=P1_MINRADIUS_DEG, maxradius=P1_MAXRADIUS_DEG,
    channel="BH?", starttime=SRC["time"], endtime=SRC["time"] + P1_DURATION_S,
    level="channel",
)

rows = []
for net in inv_all:
    for sta in net:
        comps = {ch.code[-1] for ch in sta.channels}
        horizontal = {"N", "E"} <= comps or {"1", "2"} <= comps
        if "Z" not in comps or not horizontal:
            continue
        if len(net.code) > 2 or len(sta.code) > 5:
            continue  # Syngine limits the code lengths of synthetic traces
        dist_m, az, baz = gps2dist_azimuth(SRC["lat"], SRC["lon"], sta.latitude, sta.longitude)
        rows.append(dict(net=net.code, sta=sta.code, lat=sta.latitude, lon=sta.longitude,
                         dist_km=dist_m / 1e3, az=az, baz=baz))

cands = pd.DataFrame(rows).drop_duplicates(["net", "sta"])
if cands.empty:
    raise LookupError("No three-component BH stations found. Widen P1_MAXRADIUS_DEG.")
cands["sector"] = (cands["az"] // (360.0 / P1_N_STATIONS)).astype(int)
STATIONS = (cands.sort_values("dist_km").groupby("sector").head(1)
            .sort_values("az").reset_index(drop=True))
print(f"{len(cands)} candidate stations; selected {len(STATIONS)}:")
STATIONS[["net", "sta", "lat", "lon", "dist_km", "az"]].round(2)

### 2.4 Six elementary Green's functions

This is the core step. The cell sends **six** Syngine requests. Each one has a moment tensor with exactly one non-zero element of size `M_UNIT`, so each result is one elementary Green's function g_ik (scaled by `M_UNIT`) at every selected station. The earthquake itself does not enter yet.

**[L5]** Syngine converts the geographic latitudes of source and receiver to geocentric latitudes before it computes distances. The difference is at most about 0.2° and does not matter at these periods [R2].

In [ ]:
P1_BULK = [{"latitude": r.lat, "longitude": r.lon, "networkcode": r.net, "stationcode": r.sta}
           for r in STATIONS.itertuples()]
P1_COMMON = dict(
    model=MODEL,
    sourcelatitude=SRC["lat"], sourcelongitude=SRC["lon"],
    sourcedepthinmeters=round(SRC["depth_km"] * 1000.0),
    origintime=SRC["time"], endtime=P1_DURATION_S,
    components="ZRT", units="displacement",
)

ELEM = {}
for k, key in enumerate(MT_KEYS):
    m = np.zeros(6)
    m[k] = M_UNIT
    ELEM[key] = syngine_bulk(P1_BULK, sourcemomenttensor=m, **P1_COMMON)
    print(f"{key}: {len(ELEM[key])} traces")

tr0 = ELEM["Mrr"][0]
P1_DT, P1_N = tr0.stats.delta, tr0.stats.npts
P1_T = np.arange(P1_N) * P1_DT
for key in MT_KEYS:
    for tr in ELEM[key]:
        if tr.stats.npts != P1_N or abs(tr.stats.delta - P1_DT) > 1e-9:
            raise ValueError(f"Inconsistent sampling in elementary GF {key} {tr.id}")
print(f"Sampling interval {P1_DT} s, {P1_N} samples")

In [ ]:
# Gallery of the six elementary Green's functions at the nearest station.
# The shapes differ because each element radiates a different pattern of P, SV and SH energy.
sta0 = STATIONS.sort_values("dist_km").iloc[0]
fig, axes = plt.subplots(6, 3, figsize=(12, 10), sharex=True)
for row, key in enumerate(MT_KEYS):
    for col, comp in enumerate("ZRT"):
        g = pick(ELEM[key], sta0.sta, comp).data / M_UNIT
        axes[row, col].plot(P1_T, g, lw=0.8, color="k")
        axes[row, col].set_ylabel(key if col == 0 else "")
        if row == 0:
            axes[row, col].set_title(f"{sta0.net}.{sta0.sta} {comp}  (m per N m)")
for ax in axes[-1]:
    ax.set_xlabel("Time after origin (s)")
fig.suptitle(f"Elementary Green's functions, {MODEL}, distance {sta0.dist_km:.0f} km")
fig.tight_layout()

### 2.5 [C1] Superposition check

Equation (9) of the lecture claims that the seismogram is a linear combination of Green's functions. The next cell tests this against the service. It requests the synthetic for the full moment tensor directly and compares it with the sum Σ (M_k / M_UNIT) · (elementary response k).

The two must agree to within the numerical precision of the stored data. When they do not agree, the error message lists the traces that failed.

In [ ]:
def superpose(elem, mt, station, comp):
    return sum((mt[k] / M_UNIT) * pick(elem[key], station, comp).data.astype(float)
               for k, key in enumerate(MT_KEYS))


DIRECT = syngine_bulk(P1_BULK, sourcemomenttensor=SRC["mt"], **P1_COMMON)

rows, failures = [], []
for r in STATIONS.itertuples():
    for comp in "ZRT":
        u_sup = superpose(ELEM, SRC["mt"], r.sta, comp)
        u_dir = pick(DIRECT, r.sta, comp).data.astype(float)
        rel = np.max(np.abs(u_sup - u_dir)) / np.max(np.abs(u_dir))
        rows.append(dict(station=f"{r.net}.{r.sta}", comp=comp, max_rel_error=rel))
        if rel > 1e-3:
            failures.append((r.sta, comp, rel))
check_c1 = pd.DataFrame(rows)
if failures:
    raise AssertionError(f"[C1] Superposition failed for {failures}. Syngine output is not linear in the moment tensor, "
                         "or the component order differs from Mrr, Mtt, Mpp, Mrt, Mrp, Mtp.")
print(f"[C1] passed: largest relative error {check_c1.max_rel_error.max():.1e}")

### 2.6 Source time function

Syngine removes the source time function of the AxiSEM run, so a raw Syngine trace approximates the response to a **step** in moment, limited to the resolution of the model [R2]. A real earthquake releases its moment over a finite time. Convolution with a moment-rate function s(t) of unit area adds that duration.

**[A2]** A symmetric triangle centered on the centroid time, with the half duration from Section 2.2. At periods longer than 20 s, the exact shape of a 5 to 6 s source does not matter much.

In [ ]:
def triangle_stf(half_duration, dt):
    # Unit-area triangle on [0, 2h], sampled at dt. Returns (stf, index of the peak).
    n = max(2, int(np.ceil(2 * half_duration / dt)))
    t = np.arange(n + 1) * (2 * half_duration / n)
    s = np.clip(1 - np.abs(t - half_duration) / half_duration, 0, None)
    s_dt = np.interp(np.arange(0, 2 * half_duration + dt / 2, dt), t, s)
    s_dt /= s_dt.sum() * dt
    return s_dt, int(round(half_duration / dt))


def convolve_stf(g, stf, dt, center_index=0):
    # Convolution with a unit-area STF. center_index shifts the STF so that
    # sample center_index sits at t = 0 (0 = causal STF).
    full = signal.fftconvolve(g, stf)
    return full[center_index:center_index + len(g)] * dt


STF1, STF1_C = triangle_stf(SRC["half_duration_s"], P1_DT)
SYN = {}
for r in STATIONS.itertuples():
    for comp in "ZRT":
        SYN[(r.sta, comp)] = convolve_stf(superpose(ELEM, SRC["mt"], r.sta, comp), STF1, P1_DT, STF1_C)
print(f"Triangle STF: {len(STF1)} samples, area {STF1.sum() * P1_DT:.6f}")

### 2.7 [C2] Source-time-function check (optional)

Section 2.6 assumes that a raw Syngine trace is a step response with no additional smoothing. Syngine can also convolve a custom source time function on the server [R1]. This cell requests that for one station and compares it with the local convolution. When the two differ, the step-response assumption is wrong and the error message says so.

The check sends one extra request. Set `RUN_C2 = False` to skip it.

In [ ]:
RUN_C2 = True


def syngine_get_raw(params):
    # Direct GET for parameters that the ObsPy client does not expose (custom STF).
    key = CACHE_DIR / f"{_cache_key({'raw': params})}.mseed"
    if key.exists():
        return read(str(key))
    url = syn.base_url.rstrip("/") + "/query"
    r = requests.get(url, params=params, timeout=SYNGINE_TIMEOUT_S)
    if r.status_code != 200:
        raise RuntimeError(f"Syngine GET failed with HTTP {r.status_code}: {r.text[:500]}")
    if r.content[:2] == b"PK":  # ZIP archive
        st = Stream()
        with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
            for name in zf.namelist():
                st += read(io.BytesIO(zf.read(name)))
    else:
        st = read(io.BytesIO(r.content))
    st.write(str(key), format="MSEED")
    return st


if RUN_C2:
    h = SRC["half_duration_s"]
    n_stf = max(4, int(np.ceil(2 * h)))            # about 1 s spacing
    dts = 2 * h / n_stf
    t_stf = np.arange(n_stf + 1) * dts
    stf_vals = np.clip(1 - np.abs(t_stf - h) / h, 0, None)   # starts and ends with 0, as the service requires
    params = {
        "model": MODEL, "format": "miniseed", "nodata": 404,
        "receiverlatitude": sta0.lat, "receiverlongitude": sta0.lon,
        "networkcode": sta0.net, "stationcode": sta0.sta,
        "sourcelatitude": SRC["lat"], "sourcelongitude": SRC["lon"],
        "sourcedepthinmeters": round(SRC["depth_km"] * 1000.0),
        "sourcemomenttensor": ",".join(f"{x:.6e}" for x in SRC["mt"]),
        "origintime": SRC["time"].strftime("%Y-%m-%dT%H:%M:%S.%f"),
        "endtime": P1_DURATION_S, "components": "ZRT", "units": "displacement",
        "cstf-relative-origin-time-in-sec": f"{h:.4f}",
        "cstf-sample-spacing-in-sec": f"{dts:.6f}",
        "cstf-data": ",".join(f"{v:.6f}" for v in stf_vals),
    }
    server = syngine_get_raw(params)
    for comp in "ZRT":
        a = SYN[(sta0.sta, comp)]
        b = pick(server, sta0.sta, comp).data.astype(float)
        n = min(len(a), len(b))
        a, b = a[:n], b[:n]
        cc = np.dot(a, b) / np.sqrt(np.dot(a, a) * np.dot(b, b))
        ratio = np.max(np.abs(a)) / np.max(np.abs(b))
        print(f"[C2] {sta0.sta}.{comp}: correlation {cc:.4f}, amplitude ratio (local/server) {ratio:.3f}")
        if cc < 0.99 or abs(ratio - 1) > 0.05:
            raise AssertionError(
                "[C2] Local and server-side STF convolution differ. Raw Syngine traces are not plain step "
                "responses, or the server centers the STF differently. Do not trust Section 2.6 until this is resolved."
            )
    print("[C2] passed")

### 2.8 Recorded seismograms

For each selected station, the cell downloads the 2001 recording, removes the instrument response to displacement, rotates to vertical, radial, and transverse (Z, R, T), and applies the same band-pass filter as the synthetics. A station that fails (for example, a gap or missing response) is reported by name with the reason, and is left out of the comparison.

In [ ]:
PRE_FILT = (0.004, 0.008, 0.2, 0.4)   # Hz, taper for response removal


def bandpass_array(x, dt, band_s):
    tr = Trace(data=np.asarray(x, dtype=float), header={"delta": dt})
    tr.taper(0.05)
    tr.filter("bandpass", freqmin=1 / band_s[1], freqmax=1 / band_s[0], corners=4, zerophase=True)
    return tr.data


def fetch_observed(row):
    t0 = SRC["time"]
    inv = fdsn.get_stations(network=row.net, station=row.sta, channel="BH?",
                            starttime=t0, endtime=t0 + P1_DURATION_S, level="response")
    st = fdsn.get_waveforms(row.net, row.sta, "*", "BH?", t0 - P1_PAD_S, t0 + P1_DURATION_S + P1_PAD_S)
    locs = sorted({tr.stats.location for tr in st}, key=lambda loc: (loc not in ("00", ""), loc))
    st = st.select(location=locs[0])
    st.merge()
    if any(isinstance(tr.data, np.ma.MaskedArray) for tr in st):
        raise ValueError("data gap in the time window")
    st.detrend("demean")
    st.detrend("linear")
    st.taper(0.05)
    st.remove_response(inventory=inv, output="DISP", pre_filt=PRE_FILT)
    st.rotate("->ZNE", inventory=inv)
    st.rotate("NE->RT", back_azimuth=row.baz)
    out = {}
    for comp in "ZRT":
        tr = pick(st, row.sta, comp)
        x = bandpass_array(tr.data, tr.stats.delta, P1_PERIOD_BAND_S)
        t_rel = tr.times() + (tr.stats.starttime - t0)
        out[comp] = np.interp(P1_T, t_rel, x)       # onto the synthetic time axis
    return out


OBS, dropped = {}, []
for r in STATIONS.itertuples():
    try:
        OBS[r.sta] = fetch_observed(r)
        print(f"{r.net}.{r.sta}: OK (location code, response and rotation applied)")
    except Exception as err:
        dropped.append((f"{r.net}.{r.sta}", repr(err)))
        print(f"{r.net}.{r.sta}: DROPPED: {err!r}")
if not OBS:
    raise RuntimeError(f"No station produced usable data: {dropped}")

In [ ]:
def compare(obs, syn, dt, max_lag_s=20.0):
    cc_full = signal.correlate(obs, syn, mode="full")
    lags = signal.correlation_lags(len(obs), len(syn)) * dt
    norm = np.sqrt(np.dot(obs, obs) * np.dot(syn, syn))
    keep = np.abs(lags) <= max_lag_s
    best = np.argmax(cc_full[keep])
    return dict(cc0=np.dot(obs, syn) / norm, cc_best=cc_full[keep][best] / norm,
                lag_s=lags[keep][best], amp_ratio=np.max(np.abs(syn)) / np.max(np.abs(obs)))


used = STATIONS[STATIONS.sta.isin(list(OBS))].reset_index(drop=True)
fig, axes = plt.subplots(len(used), 3, figsize=(14, 2.1 * len(used)), sharex=True, squeeze=False)
metrics = []
for i, r in enumerate(used.itertuples()):
    for j, comp in enumerate("ZRT"):
        o = OBS[r.sta][comp] * 1e6
        s = bandpass_array(SYN[(r.sta, comp)], P1_DT, P1_PERIOD_BAND_S) * 1e6
        m = compare(o, s, P1_DT)
        metrics.append(dict(station=f"{r.net}.{r.sta}", dist_km=round(r.dist_km), az=round(r.az), comp=comp, **m))
        ax = axes[i, j]
        ax.plot(P1_T, o, "k", lw=0.9, label="recorded")
        ax.plot(P1_T, s, "r", lw=0.9, label="synthetic")
        ax.set_title(f"{r.net}.{r.sta} {comp}  {r.dist_km:.0f} km, az {r.az:.0f}°  cc={m['cc0']:.2f}", fontsize=9)
        if j == 0:
            ax.set_ylabel("µm")
axes[0, 0].legend(loc="upper right", fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("Time after origin (s)")
fig.suptitle(f"2001 Nisqually: recorded vs forward-modeled displacement, {P1_PERIOD_BAND_S[0]:.0f}-{P1_PERIOD_BAND_S[1]:.0f} s")
fig.tight_layout()

METRICS = pd.DataFrame(metrics)
METRICS.round(2)

### 2.9 Reading the comparison

The table reports, per trace: the zero-lag correlation `cc0`; the best correlation within ±20 s (`cc_best`) and its lag (`lag_s` > 0 means that the recorded wave arrives later than the synthetic); and the amplitude ratio synthetic/recorded.

What to expect:

- **Amplitudes within a factor of about 2** confirm that the moment, the units (meters), and the response removal are consistent. A ratio near 10⁹ or 10⁻⁹ indicates a unit error (nanometers against meters).
- **Good correlation on most traces** confirms the mechanism. The transverse component is the most sensitive to the strike of the fault.
- **Late recorded surface waves** (positive lags that grow with distance) are normal. ak135f is a global average model. The crust under the Pacific Northwest, and the thick sediments of the Puget Lowland, are slower than that average at these periods.

**[L1]** A 1D model has no subducting slab, no 3D crust, and no sedimentary basins. The M9 Project model of Cascadia includes the Seattle basin, and its simulations show basin amplification of a factor of 2 to 5 at periods of 1 to 10 s [R8]. This notebook cannot reproduce that. Its purpose is to show the forward method, not to predict shaking at short periods.

This is the forward model working as the lecture describes. Going backward (the inversion) uses the same six elementary Green's functions: with u and g_ik known, solve u = Σ M_k (g_ik * s) for the six M_k by least squares. That is how regional moment tensors are computed [R13].

## 3. Part 2: a full-margin M9 megathrust scenario

### 3.1 Plate-interface geometry (Slab2)

The subfaults sit on the Slab2 model of the Cascadia plate interface [R7]. The cell reads the file list of the ScienceBase item, downloads the depth, strike, and dip grids, and caches them in `DATA_DIR`.

If the download fails (for example, when ScienceBase is unavailable), download the three `cas_slab2_*.grd` files by hand from the ScienceBase item and put them in `DATA_DIR`. The cell uses local files first.

**[L2]** The M9 Project used a smoothed version of the McCrory et al. (2012) interface, not Slab2 [R8]. The two agree in general shape but differ in detail.

In [ ]:
def sciencebase_files(item_id):
    r = requests.get(f"https://www.sciencebase.gov/catalog/item/{item_id}", params={"format": "json"}, timeout=60)
    r.raise_for_status()
    found = {}

    def walk(o):
        if isinstance(o, dict):
            if isinstance(o.get("name"), str) and isinstance(o.get("url"), str):
                found[o["name"]] = o["url"]
            for v in o.values():
                walk(v)
        elif isinstance(o, list):
            for v in o:
                walk(v)

    walk(r.json())
    return found


def get_slab2_grid(kind):
    local = sorted(DATA_DIR.glob(f"cas_slab2_{kind}_*.grd"))
    if local:
        return local[-1]
    files = sciencebase_files(SLAB2_ITEM_ID)
    matches = sorted(n for n in files if n.startswith(f"cas_slab2_{kind}_") and n.endswith(".grd"))
    if not matches:
        raise FileNotFoundError(f"No cas_slab2_{kind}_*.grd in ScienceBase item {SLAB2_ITEM_ID}. Files: {sorted(files)}")
    path = DATA_DIR / matches[-1]
    with requests.get(files[matches[-1]], stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    return path


def open_grid(path):
    errors = {}
    for engine in ("netcdf4", "h5netcdf", "scipy"):
        try:
            ds = xr.open_dataset(path, engine=engine)
            break
        except Exception as err:
            errors[engine] = repr(err)
    else:
        raise RuntimeError(f"Could not open {path} with any netCDF engine: {errors}")
    da = ds[list(ds.data_vars)[0]]
    xdim = next(d for d in da.dims if d.lower() in ("x", "lon", "longitude"))
    ydim = next(d for d in da.dims if d.lower() in ("y", "lat", "latitude"))
    lon = da[xdim].values.astype(float)
    lon = np.where(lon > 180, lon - 360, lon)
    return lon, da[ydim].values.astype(float), da.transpose(ydim, xdim).values.astype(float)


grids = {kind: open_grid(get_slab2_grid(kind)) for kind in ("dep", "str", "dip")}
SLAB_LON, SLAB_LAT, SLAB_DEP = grids["dep"]
for kind in ("str", "dip"):
    lon_k, lat_k, _ = grids[kind]
    if lon_k.shape != SLAB_LON.shape or not (np.allclose(lon_k, SLAB_LON) and np.allclose(lat_k, SLAB_LAT)):
        raise ValueError(f"Slab2 {kind} grid is not on the same grid as the depth grid")
SLAB_DEP = np.abs(SLAB_DEP)          # Slab2 stores depth as negative km
SLAB_STR, SLAB_DIP = grids["str"][2], grids["dip"][2]
print(f"Slab2 grid: {SLAB_LON.min():.2f} to {SLAB_LON.max():.2f} E, {SLAB_LAT.min():.2f} to {SLAB_LAT.max():.2f} N, "
      f"depth {np.nanmin(SLAB_DEP):.1f} to {np.nanmax(SLAB_DEP):.1f} km")

### 3.2 Subfaults

The fault is divided into latitude bands of `DLAT` (about 44 km) and depth rows given by `DEPTH_EDGES_KM`. Each subfault takes the mean position, depth, strike, and dip of the Slab2 cells in its band and row. Its along-strike length is DLAT × 111.19 km / |cos(strike)|, and its down-dip width is its depth extent divided by sin(dip).

- **[A4]** 40.4°N to 50.0°N gives a rupture about 1070 km long, close to the 1100 km preferred for 1700 by Satake et al. (2003) [R6].
- **[A5]** The down-dip limit of slip controls shaking strongly, and it is not well known. The M9 Project tested three limits, from a GPS locking contour to the top of the tremor zone [R8, R9]. 25 km is a middle value. Change `DEPTH_EDGES_KM` to test it.
- **[A12] / [L3]** Each subfault is a point source. That is adequate for periods much longer than the time the rupture needs to cross it (about 44 km / 2.5 km/s ≈ 18 s), and when the subfault spacing is shorter than about half a surface-wave wavelength (about 45 km at 30 s). This is why the scenario is low-passed at 30 s.

In [ ]:
LON2D, LAT2D = np.meshgrid(SLAB_LON, SLAB_LAT)
n_lat = int(round((LAT_MAX - LAT_MIN) / DLAT))
n_dep = len(DEPTH_EDGES_KM) - 1
if len(SLIP_DOWNDIP_WEIGHTS) != n_dep:
    raise ValueError("SLIP_DOWNDIP_WEIGHTS needs one value per depth row")

rows, skipped = [], []
for i in range(n_lat):
    la0 = LAT_MIN + i * DLAT
    band = (LAT2D >= la0) & (LAT2D < la0 + DLAT)
    for j in range(n_dep):
        d0, d1 = DEPTH_EDGES_KM[j], DEPTH_EDGES_KM[j + 1]
        m = band & (SLAB_DEP >= d0) & (SLAB_DEP < d1) & np.isfinite(SLAB_STR) & np.isfinite(SLAB_DIP)
        if m.sum() < 3:
            skipped.append((i, j))
            continue
        strike = np.degrees(np.angle(np.mean(np.exp(1j * np.radians(SLAB_STR[m]))))) % 360
        dip = float(np.mean(SLAB_DIP[m]))
        width_km = (SLAB_DEP[m].max() - SLAB_DEP[m].min()) / np.sin(np.radians(dip))
        length_km = DLAT * 111.19 / max(abs(np.cos(np.radians(strike))), 0.5)
        rows.append(dict(i=i, j=j, lat=LAT2D[m].mean(), lon=LON2D[m].mean(), depth_km=SLAB_DEP[m].mean(),
                         strike=round(strike, 1), dip=round(dip, 1),
                         length_km=length_km, width_km=width_km, area_m2=length_km * width_km * 1e6))
SUB = pd.DataFrame(rows)
SUB = SUB[SUB.width_km > 1.0].reset_index(drop=True)
print(f"{len(SUB)} subfaults ({n_lat} bands x {n_dep} rows; {len(skipped)} empty cells skipped)")
print(f"Total area {SUB.area_m2.sum() / 1e6:.3g} km^2; strike {SUB.strike.min():.0f}-{SUB.strike.max():.0f} deg; "
      f"dip {SUB.dip.min():.1f}-{SUB.dip.max():.1f} deg")
SUB.head()

### 3.3 Slip, rise time, and rupture time

- **[A6]** Slip is tapered at both ends of the rupture and is larger at shallow depth. The M9 Project used large, long-duration slip at shallow depth with high-stress-drop M8 subevents deeper on the fault [R8]. This notebook models only the background slip, because the subevents radiate mainly at periods shorter than 30 s. A smoothed log-normal factor adds heterogeneity with a fixed seed, so runs are repeatable.
- **[A7]** Slip is scaled so that the total moment gives Mw 9.0 with a rigidity of 30 GPa. **[L4]** Rigidity on real megathrusts is lower at shallow depth. With a constant rigidity, the shallow slip needed for a given moment is too small.
- **[A8]** Every subfault is a pure thrust (rake 90°). The real slip direction varies along the margin.
- **[A9]** Rupture spreads from the hypocenter at a constant 2.5 km/s. The M9 Project found the average rupture velocity to be one of the parameters that control shaking [R9].
- **[A10]** Rise time = slip / 0.65 m/s, limited to 30 s. These are the background-slip values of the M9 Project [R8]. Each subfault's moment-rate function is a boxcar of the rise time convolved with a boxcar of the rupture crossing time √(area) / v_r.

In [ ]:
def boxcar(duration, dt):
    n = max(1, int(round(duration / dt)))
    return np.ones(n) / (n * dt)


def subfault_stfs(slip_m, area_m2, dt):
    stfs = []
    for s, a in zip(slip_m, area_m2):
        rise = np.clip(s / SLIP_VELOCITY_MS, 2.0, MAX_RISE_TIME_S)
        crossing = np.sqrt(a) / 1e3 / RUPTURE_VELOCITY_KMS
        stf = np.convolve(boxcar(rise, dt), boxcar(crossing, dt)) * dt
        stfs.append(stf / (stf.sum() * dt))
    return stfs


def rupture_times(sub, hypo_idx):
    h = sub.loc[hypo_idx]
    dx = (sub.lon - h.lon) * 111.19 * np.cos(np.radians(h.lat))
    dy = (sub.lat - h.lat) * 111.19
    dz = sub.depth_km - h.depth_km
    return np.sqrt(dx**2 + dy**2 + dz**2).values / RUPTURE_VELOCITY_KMS


def nearest_subfault(sub, lat, depth_km):
    d = np.hypot((sub.lat - lat) * 111.19, sub.depth_km - depth_km)
    return int(np.argmin(d.values))


# Slip shape: taper along strike x down-dip weights x smoothed log-normal noise.
xi = (SUB.i.values + 0.5) / n_lat
taper = np.where(xi < 0.15, 0.5 * (1 - np.cos(np.pi * xi / 0.15)),
                 np.where(xi > 0.85, 0.5 * (1 - np.cos(np.pi * (1 - xi) / 0.15)), 1.0))
shape = (0.15 + 0.85 * taper) * np.asarray(SLIP_DOWNDIP_WEIGHTS)[SUB.j.values]
if RANDOM_SLIP_SIGMA > 0:
    rng = np.random.default_rng(RANDOM_SEED)
    noise = rng.normal(size=(n_lat, n_dep))
    kernel = np.exp(-0.5 * (np.arange(-3, 4) / 1.5) ** 2)
    noise = np.apply_along_axis(lambda c: np.convolve(c, kernel / kernel.sum(), mode="same"), 0, noise)
    noise /= noise.std()
    shape *= np.exp(RANDOM_SLIP_SIGMA * noise[SUB.i.values, SUB.j.values])

M0_TARGET = m0_from_mw(TARGET_MW)
SUB["slip_m"] = shape * M0_TARGET / (RIGIDITY_PA * np.sum(shape * SUB.area_m2.values))
SUB["m0"] = RIGIDITY_PA * SUB.area_m2 * SUB.slip_m
HYPO_IDX = {name: nearest_subfault(SUB, lat, dep) for name, (lat, dep) in HYPOCENTERS.items()}
for name, idx in HYPO_IDX.items():
    SUB[f"t_rup_{name}"] = rupture_times(SUB, idx)

avg_slip = SUB.m0.sum() / (RIGIDITY_PA * SUB.area_m2.sum())
print(f"Mw {mw_from_m0(SUB.m0.sum()):.2f}; M0 {SUB.m0.sum():.2e} N m")
print(f"Rupture length {(LAT_MAX - LAT_MIN) * 111.19:.0f} km; average slip {avg_slip:.1f} m; peak slip {SUB.slip_m.max():.1f} m")
print("  For comparison, Satake et al. (2003): 1100 km, average slip about 14 m, M0 about 5e22 N m [R6]")
for name, idx in HYPO_IDX.items():
    print(f"Hypocenter '{name}': subfault {idx} at {SUB.lat[idx]:.2f} N, {SUB.depth_km[idx]:.1f} km; "
          f"last subfault starts at {SUB[f't_rup_{name}'].max():.0f} s")

In [ ]:
def base_map(ax_extent=(-130.5, -119.5, 39.5, 51.0)):
    if HAVE_CARTOPY:
        fig = plt.figure(figsize=(7, 9))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent(ax_extent)
        ax.add_feature(cfeature.LAND, color="0.93")
        ax.add_feature(cfeature.COASTLINE, lw=0.6)
        ax.add_feature(cfeature.BORDERS, lw=0.4)
        ax.add_feature(cfeature.STATES, lw=0.3)
        ax.gridlines(draw_labels=True, lw=0.2)
        return fig, ax, {"transform": ccrs.PlateCarree()}
    fig, ax = plt.subplots(figsize=(7, 9))
    ax.set_xlim(ax_extent[:2]); ax.set_ylim(ax_extent[2:])
    ax.set_aspect(1 / np.cos(np.radians(45)))
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    return fig, ax, {}


fig, ax, kw = base_map()
sc = ax.scatter(SUB.lon, SUB.lat, c=SUB.slip_m, s=SUB.area_m2 / 2e7, cmap="viridis", marker="s", **kw)
plt.colorbar(sc, ax=ax, shrink=0.6, label="Slip (m)")
cs = ax.contour(SLAB_LON, SLAB_LAT, SLAB_DEP, levels=[10, 20, 30, 40, 60], colors="0.4", linewidths=0.5, **kw)
ax.clabel(cs, fmt="%d km", fontsize=7)
for name, idx in HYPO_IDX.items():
    ax.plot(SUB.lon[idx], SUB.lat[idx], "r*", ms=16, **kw)
    ax.text(SUB.lon[idx] - 0.3, SUB.lat[idx], f"{name}\nhypocenter", ha="right", fontsize=8, color="r", **kw)
ax.plot(RECEIVERS.lon, RECEIVERS.lat, "k^", ms=8, **kw)
for r in RECEIVERS.itertuples():
    ax.text(r.lon + 0.15, r.lat, r.name, fontsize=8, **kw)
ax.set_title(f"M{TARGET_MW} scenario: {len(SUB)} subfaults on Slab2 (contours: interface depth)")

### 3.4 The Green's-function bank

One Syngine request per subfault. Each request computes the response of all six receivers to a double couple with that subfault's strike, dip, and rake, with a moment of `M_UNIT`, at a fixed reference time. The result is a bank G[subfault, receiver, component, time]. It holds **no information about slip, timing, or hypocenter**. That information enters only in the sum of Section 3.5.

The first run sends about 100 requests (a few minutes). Later runs read the cache.

In [ ]:
REF_TIME = UTCDateTime("2000-01-01T00:00:00")
P2_BULK = [{"latitude": r.lat, "longitude": r.lon, "networkcode": "SC", "stationcode": r.code}
           for r in RECEIVERS.itertuples()]
COMPS = "ZNE"


def fetch_subfault_gf(row):
    return row.Index, syngine_bulk(
        P2_BULK, model=MODEL,
        sourcelatitude=round(row.lat, 4), sourcelongitude=round(row.lon, 4),
        sourcedepthinmeters=round(row.depth_km * 1000.0),
        sourcedoublecouple=[row.strike, row.dip, RAKE_DEG, M_UNIT],
        origintime=REF_TIME, endtime=P2_REQUEST_S, components=COMPS, units="displacement",
    )


t_start = time.time()
streams = {}
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = [pool.submit(fetch_subfault_gf, row) for row in SUB.itertuples()]
    for n_done, fut in enumerate(as_completed(futures), 1):
        idx, st = fut.result()                # re-raises any request error
        streams[idx] = st
        if n_done % 10 == 0 or n_done == len(futures):
            print(f"{n_done}/{len(futures)} subfaults ({time.time() - t_start:.0f} s)")

P2_DT = streams[0][0].stats.delta
lengths = {tr.stats.npts for st in streams.values() for tr in st}
if max(lengths) - min(lengths) > 2 or any(abs(tr.stats.delta - P2_DT) > 1e-9 for st in streams.values() for tr in st):
    raise ValueError(f"Green's functions have inconsistent sampling: lengths {sorted(lengths)}")
P2_N = min(lengths)
BANK = np.zeros((len(SUB), len(RECEIVERS), len(COMPS), P2_N))
for i, st in streams.items():
    for r, code_ in enumerate(RECEIVERS.code):
        for c, comp in enumerate(COMPS):
            BANK[i, r, c] = pick(st, code_, comp).data[:P2_N]
P2_T = np.arange(P2_N) * P2_DT
print(f"Bank shape {BANK.shape}, dt {P2_DT} s, {BANK.nbytes / 1e6:.0f} MB")

### 3.5 Equation (7) as code

`synthesize` evaluates u(x, t) = Σ_j (M0_j / M_UNIT) · [G_j(x, t − T_j) * s_j(t)]. The three operations map one to one onto the lecture:

1. `fftconvolve(..., stf)`: the time integral over the source history (the dt′ integral).
2. The shift by T_j: the rupture reaching subfault j later. The shift uses linear interpolation between samples.
3. The sum over j: the volume integral over the source.

**[C3]** The moment-rate function of the whole rupture, Σ M0_j s_j(t − T_j), must integrate to the target moment.

In [ ]:
def delay(x, delay_s, dt):
    # y(t) = x(t - delay) along the last axis, zero before the delay.
    k = delay_s / dt
    k0, f = int(np.floor(k)), k - np.floor(k)
    n = x.shape[-1]
    a = np.zeros_like(x)
    b = np.zeros_like(x)
    if k0 < n:
        a[..., k0:] = x[..., :n - k0]
    if k0 + 1 < n:
        b[..., k0 + 1:] = x[..., :n - k0 - 1]
    return (1 - f) * a + f * b


def synthesize(bank, dt, m0, t_rup, stfs):
    out = np.zeros(bank.shape[1:])
    for j in range(bank.shape[0]):
        g = signal.fftconvolve(bank[j], stfs[j][None, None, :], axes=-1)[..., :bank.shape[-1]] * dt
        out += (m0[j] / M_UNIT) * delay(g, t_rup[j], dt)
    return out


def moment_rate(dt, n, m0, t_rup, stfs):
    mrf = np.zeros(n)
    for j in range(len(m0)):
        s = np.zeros(n)
        s[:min(n, len(stfs[j]))] = stfs[j][:n]
        mrf += m0[j] * delay(s, t_rup[j], dt)
    return mrf


def lowpass(x, dt, period_s):
    sos = signal.butter(4, (1.0 / period_s) / (0.5 / dt), btype="low", output="sos")
    return signal.sosfiltfilt(sos, x, axis=-1)


STFS = subfault_stfs(SUB.slip_m.values, SUB.area_m2.values, P2_DT)
RESULTS = {}
for name in HYPOCENTERS:
    t_rup = SUB[f"t_rup_{name}"].values
    u = lowpass(synthesize(BANK, P2_DT, SUB.m0.values, t_rup, STFS), P2_DT, P2_MIN_PERIOD_S)
    RESULTS[name] = {"disp": u, "vel": np.gradient(u, P2_DT, axis=-1),
                     "mrf": moment_rate(P2_DT, P2_N, SUB.m0.values, t_rup, STFS)}

    # [C3] Moment conservation
    m0_mrf = RESULTS[name]["mrf"].sum() * P2_DT
    if abs(m0_mrf / M0_TARGET - 1) > 1e-2:
        raise AssertionError(f"[C3] Moment-rate function of '{name}' integrates to {m0_mrf:.3e}, "
                             f"target {M0_TARGET:.3e}. Increase P2_REQUEST_S or check the STFs.")
    print(f"[C3] '{name}': moment-rate integral {m0_mrf:.3e} N m (target {M0_TARGET:.3e})")

valid_until = P2_REQUEST_S - max(SUB[f"t_rup_{n}"].max() for n in HYPOCENTERS)
print(f"Every subfault contributes fully up to t = {valid_until:.0f} s; plots stop at {P2_PLOT_S:.0f} s.")
if valid_until < P2_PLOT_S * 0.8:
    print("Warning: late parts of the plotted records lack contributions from the last subfaults. Increase P2_REQUEST_S.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
for name, res in RESULTS.items():
    ax.plot(P2_T, res["mrf"] / 1e20, label=f"{name} hypocenter")
ax.set_xlim(0, P2_PLOT_S)
ax.set_xlabel("Time after rupture start (s)")
ax.set_ylabel("Moment rate (10²⁰ N m/s)")
ax.set_title("Moment-rate function of the scenario rupture (same slip, different hypocenter)")
ax.legend()
fig.tight_layout()

In [ ]:
res = RESULTS["south"]
fig, axes = plt.subplots(len(RECEIVERS), 3, figsize=(14, 2.0 * len(RECEIVERS)), sharex=True, squeeze=False)
for r, rec in enumerate(RECEIVERS.itertuples()):
    for c, comp in enumerate(COMPS):
        ax = axes[r, c]
        ax.plot(P2_T, res["disp"][r, c], "k", lw=0.8)
        ax.set_xlim(0, P2_PLOT_S)
        ax.set_title(f"{rec.name}  {comp}", fontsize=9)
        if c == 0:
            ax.set_ylabel("m")
for ax in axes[-1]:
    ax.set_xlabel("Time after rupture start (s)")
fig.suptitle(f"M{TARGET_MW} scenario, south hypocenter: displacement at periods > {P2_MIN_PERIOD_S:.0f} s ({MODEL})")
fig.tight_layout()

### 3.6 Reusing the Green's functions: directivity

The practical value of Green's functions is that one bank serves every rupture on the same subfaults. The next cell compares ruptures that start at the south and at the north end. It sends **no** new requests: only the delays T_j change.

A rupture that propagates toward a site compresses the arriving energy into a shorter, stronger pulse (forward directivity). Expect larger peak velocity at Victoria and Seattle for the south hypocenter, and larger peak velocity at Crescent City for the north hypocenter. The M9 Project found the same effect, and found that directivity combines with basin amplification in the Seattle basin [R9]; the basin part is absent here [L1].

In [ ]:
rows = []
for r, rec in enumerate(RECEIVERS.itertuples()):
    row = {"site": rec.name}
    for name, res in RESULTS.items():
        h_disp = np.hypot(res["disp"][r, 1], res["disp"][r, 2])
        h_vel = np.hypot(res["vel"][r, 1], res["vel"][r, 2])
        row[f"PGD_h {name} (m)"] = h_disp.max()
        row[f"PGV_h {name} (cm/s)"] = 100 * h_vel.max()
    row["PGV ratio south/north"] = row["PGV_h south (cm/s)"] / row["PGV_h north (cm/s)"]
    rows.append(row)
DIRECTIVITY = pd.DataFrame(rows)

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for ax, site in zip(axes, ["Seattle, WA", "Crescent City, CA"]):
    r = int(np.flatnonzero(RECEIVERS.name == site)[0])
    for name, res in RESULTS.items():
        ax.plot(P2_T, 100 * res["vel"][r, 1], lw=0.9, label=f"{name} hypocenter")
    ax.set_title(f"{site}: north-component velocity, periods > {P2_MIN_PERIOD_S:.0f} s", fontsize=10)
    ax.set_ylabel("cm/s")
    ax.set_xlim(0, P2_PLOT_S)
axes[0].legend()
axes[-1].set_xlabel("Time after rupture start (s)")
fig.tight_layout()
DIRECTIVITY.round(2)

**These numbers are long-period values (T > 30 s) from a 1D model.** They are not shaking-hazard estimates. The engineering-relevant motion at 0.1 to 10 s, the basin amplification, and the high-stress-drop subevents are all missing. For hazard, use the USGS ensemble ShakeMaps from the M9 Project [R9, R15].

### 3.7 Summary of limitations

| Label | Limitation | Effect |
|---|---|---|
| L1 | 1D radially symmetric Earth (ak135f) | No slab, no 3D crust, no Seattle or Tacoma basins; surface-wave timing and amplitude errors |
| L2 | Slab2 interface, not the smoothed McCrory et al. (2012) interface of the M9 Project | Small differences in depth and dip |
| L3 | Point-source subfaults about 44 km apart | Valid only at periods longer than about 30 s |
| L4 | Constant rigidity | Shallow slip is too small for a given moment |
| L5 | Geocentric latitude conversion in Syngine | At most about 0.2°; no effect at these periods |
| A5, A6, A8, A9, A11 | Down-dip limit, slip distribution, rake, rupture velocity, hypocenter | The M9 Project found that these control the ground motion; change them and compare |

## 4. Where to go next

- **Test the scenario parameters.** Change `DEPTH_EDGES_KM` (down-dip limit) or `HYPOCENTERS` and rerun from Section 3.3. A change of geometry (Section 3.2) creates new subfaults and sends new requests. A change of slip, timing, or hypocenter reuses the bank.
- **Go backward.** Use the elementary Green's functions of Section 2.4 in a least-squares moment-tensor inversion of the Nisqually records. This is the "inversion for source parameters" of the lecture.
- **Go 3D.** For realistic shaking in the Puget Lowland, replace Syngine with a 3D code and the USGS Cascadia velocity model (Stephenson et al., 2017) [R16].

## 5. References

- **[R1]** EarthScope, *IRISWS Syngine web service documentation, v1*. https://service.iris.edu/irisws/syngine/1/
- **[R2]** EarthScope, *Syngine data product description*. https://ds.iris.edu/ds/products/syngine/ ; van Driel, M., Krischer, L., Stähler, S. C., Hosseini, K., and Nissen-Meyer, T. (2015). Instaseis: instant global seismograms based on a broadband waveform database. *Solid Earth*, 6, 701–717. Nissen-Meyer, T., et al. (2014). AxiSEM: broadband 3-D seismic wavefields in axisymmetric media. *Solid Earth*, 5, 425–445.
- **[R3]** Aki, K., and Richards, P. G. (2002). *Quantitative Seismology*, 2nd ed. University Science Books. Box 4.4.
- **[R4]** Ekström, G., Nettles, M., and Dziewoński, A. M. (2012). The global CMT project 2004–2010: centroid-moment tensors for 13,017 earthquakes. *Physics of the Earth and Planetary Interiors*, 200–201, 1–9.
- **[R5]** Ichinose, G. A., Thio, H. K., and Somerville, P. G. (2004). Rupture process and near-source shaking of the 1965 Seattle-Tacoma and 2001 Nisqually, intraslab earthquakes. *Geophysical Research Letters*, 31, L10604. https://doi.org/10.1029/2004GL019668
- **[R6]** Satake, K., Wang, K., and Atwater, B. F. (2003). Fault slip and seismic moment of the 1700 Cascadia earthquake inferred from Japanese tsunami descriptions. *Journal of Geophysical Research*, 108(B11), 2535. https://doi.org/10.1029/2003JB002521
- **[R7]** Hayes, G. P., et al. (2018). Slab2, a comprehensive subduction zone geometry model. *Science*, 362, 58–61. Data: Hayes, G. (2018), USGS data release, https://doi.org/10.5066/F7PV6JNV (Cascadia item: https://www.sciencebase.gov/catalog/item/5aa312cde4b0b1c392ea3ef5)
- **[R8]** Frankel, A., Wirth, E., Marafi, N., Vidale, J., and Stephenson, W. (2018). Broadband synthetic seismograms for magnitude 9 earthquakes on the Cascadia megathrust based on 3D simulations and stochastic synthetics, Part 1: Methodology and overall results. *BSSA*, 108(5A), 2347–2369. https://doi.org/10.1785/0120180034 ; see also the 2018 NSHM workshop presentation, https://earthquake.usgs.gov/static/lfs/nshm/workshops/2018update/Day_2_Frankel_M9.pdf
- **[R9]** Wirth, E. A., Frankel, A. D., Marafi, N., Vidale, J. E., and Stephenson, W. J. (2018). Broadband synthetic seismograms for magnitude 9 earthquakes on the Cascadia megathrust based on 3D simulations and stochastic synthetics, Part 2: Rupture parameters and variability. *BSSA*, 108(5A), 2370–2388. https://doi.org/10.1785/0120180029
- **[R10]** USGS ComCat event `uw10530748` (2001 Nisqually). https://earthquake.usgs.gov/earthquakes/eventpage/uw10530748
- **[R11]** EERI (2001). *The Nisqually earthquake of 28 February 2001: preliminary reconnaissance report*.
- **[R12]** Pacific Northwest Seismic Network, *2001 Nisqually earthquake*. https://pnsn.org/education/pnw-earthquakes/notable/nisqually
- **[R13]** Minson, S. E., and Dreger, D. S. (2008). Stable inversions for complete moment tensors. *Geophysical Journal International*, 174(2), 585–592. https://doi.org/10.1111/j.1365-246X.2008.03797.x
- **[R14]** Silver, P. G., and Jordan, T. H. (1982). Optimal estimation of scalar seismic moment. *Geophysical Journal International*, 70, 755–787.
- **[R15]** Wirth, E. A., Grant, A., Marafi, N. A., and Frankel, A. D. (2021). Ensemble ShakeMaps for magnitude 9 earthquakes on the Cascadia subduction zone. *Seismological Research Letters*, 92, 199–211. USGS scenario catalog: https://earthquake.usgs.gov/scenarios/catalog/cszm9/
- **[R16]** Stephenson, W. J., Reitman, N. G., and Angster, S. J. (2017). P- and S-wave velocity models incorporating the Cascadia subduction zone for 3D earthquake ground motion simulations. USGS Open-File Report 2017-1152. https://doi.org/10.3133/ofr20171152
- **Lecture:** MIT OpenCourseWare 12.510 Introduction to Seismology, Lecture 17 notes (Green's functions, Section B). https://ocw.mit.edu/courses/12-510-introduction-to-seismology-spring-2010/